# ETCCDI Extreme Indices — Processing Workflow

Computes ETCCDI extreme climate indices by running each index's bash script
over every model file for every scenario.

**To run a different index or add a new one:** edit the `INDICES` dict below.
That's the only thing that needs to change — paths, index name, whether it
needs a reference period, etc. all live there.

## 1. Setup — paths and global settings

In [ ]:
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import xarray as xr

# --- Project paths -----------------------------------------------------
PROJECT_DIR = Path("/mnt/wcs-s1-d1/BOLT_ISIMIP_BCSD/VANUATU")
INPUT_DIR = PROJECT_DIR / "data/final"
OUTPUT_DIR = PROJECT_DIR / "data/interim/Indicators"
SCRIPT_DIR = PROJECT_DIR / "src/notebook/indicators/bash"

SCENARIOS = ["rcp26", "rcp85"]
REFSTART, REFEND = 1980, 2000  # baseline period for percentile-based indices


## 2. Index configuration

One entry per index. This is the only section to touch when adding a new
index or changing an existing one:

- `script` — bash script filename (resolved against `SCRIPT_DIR`)
- `input_var` — subfolder / variable used as input (e.g. `tasmax`, `tasmin`, `pr`)
- `output_tag` — text that replaces `input_var` in the output filename
- `needs_ref_period` — whether the script takes `REFSTART REFEND` as extra args

Outputs are written to `OUTPUT_DIR / <index name> / <scenario>/`.

In [ ]:
INDICES = {
    "hotdays":  dict(script="hotdays.sh",       input_var="tasmax", output_tag="hotdays", needs_ref_period=True),
    "tx90p":    dict(script="tx90p.sh",         input_var="tasmax", output_tag="tx90p",   needs_ref_period=True),
    "hwfi":     dict(script="heatwavefreq.sh",  input_var="tasmax", output_tag="hwfi",    needs_ref_period=True),
    "TXx":      dict(script="TXx.sh",           input_var="tasmax", output_tag="TXx",     needs_ref_period=False),
}


## 3. Core processing function

In [ ]:
def process_indicator(name, cfg, scenarios=SCENARIOS):
    """Run cfg['script'] over every model file, for every scenario, for one index.

    Skips files whose output already exists (and is non-empty). Returns
    (success_count, total_count).
    """
    script_path = SCRIPT_DIR / cfg["script"]
    total = success = 0

    for scenario in scenarios:
        in_dir = INPUT_DIR / scenario / cfg["input_var"]
        out_dir = OUTPUT_DIR / name / scenario
        out_dir.mkdir(parents=True, exist_ok=True)

        nc_files = sorted(in_dir.glob("*.nc"))
        print(f"\n=== {name} | {scenario}: {len(nc_files)} model files found ===")

        for infile in nc_files:
            total += 1
            out_name = infile.stem.replace(cfg["input_var"], cfg["output_tag"])
            outfile = out_dir / f"{out_name}.nc"

            if outfile.exists() and outfile.stat().st_size > 0:
                print(f"  -> Skipping (output exists): {outfile.name}")
                success += 1
                continue

            cmd = ["bash", str(script_path), str(infile), str(outfile)]
            if cfg["needs_ref_period"]:
                cmd += [str(REFSTART), str(REFEND)]

            result = subprocess.run(cmd, capture_output=True, text=True)
            if result.returncode == 0:
                print(f"  -> Saved: {outfile.name}")
                success += 1
            else:
                print(f"  ERROR processing {infile.name}:\n  {result.stderr.strip()}")

    print(f"\n{name}: {success}/{total} files processed successfully.")
    print(f"Outputs saved under: {OUTPUT_DIR / name}")
    return success, total


## 4. Run

Run a single index by name, or loop over `INDICES` to run everything.

In [ ]:
# Run one index:
process_indicator("TXx", INDICES["TXx"])

# Or run all configured indices:
# for name, cfg in INDICES.items():
#     process_indicator(name, cfg)


## 5. Visualize an index's output

In [ ]:
def plot_annual_series(data_dir, var, title, ylabel="Temperature (°C)", strip_from_label=()):
    """Plot the spatial-mean annual time series for every file in data_dir."""
    files = sorted(Path(data_dir).glob("*.nc"))

    plt.figure(figsize=(14, 6))
    for f in files:
        ds = xr.open_dataset(f)
        ts = ds[var].mean(dim=("lat", "lon"))

        label = f.stem
        for token in strip_from_label:
            label = label.replace(token, "")

        plt.plot(ds.time.dt.year, ts, label=label, linewidth=1.5)
        ds.close()

    plt.title(title)
    plt.xlabel("Year")
    plt.ylabel(ylabel)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_annual_series(
    data_dir=OUTPUT_DIR / "TXx" / "rcp26",
    var="tasmax",
    title="Annual TXx (Spatial Mean) - RCP2.6",
    strip_from_label=[
        "TXx_VAN_sim-hist-fut_3km_BC_",
        "_v1_day_1979-2100_leaf-year_rcp26",
    ],
)
